# Proyecto Final MAT281 — Predicción de Churn (versión Google Colab)

> **Cómo ejecutar este notebook en Colab:**
> 1. Ejecuta la celda de **instalación de librerías** (la primera de código).
> 2. Ejecuta la celda de **carga del dataset**: te pedirá subir el archivo
>    `WA_Fn-UseC_-Telco-Customer-Churn.csv` (descárgalo de
>    [Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)).
> 3. Luego usa *Entorno de ejecución → Ejecutar todo* para correr el proyecto completo.

---


In [ ]:
# === Instalación de librerías (solo necesario en Google Colab) ===
# Colab ya trae pandas, numpy, matplotlib, seaborn y scikit-learn.
# Instalamos/actualizamos xgboost y shap, e imbalanced-learn por si se requiere.
!pip install -q --upgrade xgboost shap imbalanced-learn
print("Librerías listas.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 11.1 MB/s eta 0:00:00
Librerías listas.


# Proyecto Final MAT281 — Predicción de Churn de Clientes (Telco)

**Curso:** MAT281 - Aplicaciones de la Matemática en la Ingeniería
**Tema:** Clasificación binaria — predicción de fuga de clientes (*Customer Churn*)
**Dataset:** [Telco Customer Churn (Kaggle / IBM)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

---


## 1. Definición del problema

En el mercado de telecomunicaciones, **retener un cliente existente es mucho más barato
que adquirir uno nuevo**. Si la empresa puede anticipar qué clientes están en riesgo de
**cancelar su contrato (churn)**, puede dirigir campañas de retención (descuentos, mejoras
de plan, contacto proactivo) hacia ese segmento, optimizando el gasto comercial.

**Variable objetivo:** `Churn` (`Yes` / `No`) → la transformamos a `1` / `0`.

**Tipo de problema:** clasificación binaria supervisada.

### ¿Qué error es más costoso?

- **Falso negativo** (decimos que el cliente se queda, pero en realidad se va): la empresa
  **pierde al cliente** y todo el ingreso futuro asociado — costoso porque adquirir un
  cliente nuevo cuesta entre 5 y 25 veces más que retenerlo.
- **Falso positivo** (decimos que el cliente se va, pero se iba a quedar): la empresa
  **gasta en una campaña de retención innecesaria** (ej. descuento) — costoso, pero mucho
  menor que perder al cliente.

**Conclusión de negocio:** el falso negativo es más caro → priorizamos **Recall** (capturar
la mayor cantidad posible de clientes que realmente se van) sin descuidar **Precision**
(no gastar el presupuesto de retención en todo el mundo). Por eso usaremos **F1-score** y
**ROC-AUC** como métricas resumen, y miraremos Recall en detalle al elegir el umbral final.


In [ ]:
# Librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report, precision_recall_curve)
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import xgboost as xgb
import shap

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42


## 2. Análisis exploratorio (EDA)

Cargamos el dataset y revisamos dimensiones, tipos de datos y calidad general.


In [ ]:
# === Carga del dataset ===
# Opción A (recomendada): subir el CSV manualmente desde tu computador.
# Descárgalo de Kaggle: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
import pandas as pd

try:
    # Si estamos en Colab, abrimos el diálogo para subir el archivo
    from google.colab import files
    import io
    print("Sube el archivo WA_Fn-UseC_-Telco-Customer-Churn.csv ...")
    uploaded = files.upload()
    nombre = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[nombre]))
except (ImportError, IndexError):
    # Fuera de Colab: lee el CSV desde la ruta local
    df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dimensiones:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# TotalCharges deberia ser numerica, pero viene como texto (problema de calidad de datos)
# algunos registros tienen un string vacio " " en vez de un numero (clientes con tenure=0,
# es decir, clientes nuevos que aun no han generado un cobro total)
print("Valores no numericos en TotalCharges:", (df["TotalCharges"].str.strip() == "").sum())
df.loc[df["TotalCharges"].str.strip() == "", ["customerID", "tenure", "TotalCharges"]]


In [ ]:
# Conversion y tratamiento: como tenure=0 implica que aun no se le ha cobrado nada,
# imputamos esos casos con 0 (es la interpretacion mas coherente con el negocio)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Eliminamos el ID (no aporta informacion predictiva)
df = df.drop(columns=["customerID"])

# Variable objetivo a binaria
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Nulos por columna:\n", df.isnull().sum().sum(), "valores nulos en total")
df.describe(include="all").T


In [ ]:
# Tasa global de churn
churn_rate = df["Churn"].mean()
print(f"Tasa global de churn: {churn_rate:.2%}")

fig, ax = plt.subplots()
df["Churn"].value_counts().plot(kind="bar", color=["#4C72B0", "#DD8452"], ax=ax)
ax.set_xticklabels(["No (0)", "Yes (1)"], rotation=0)
ax.set_title("Distribucion de la variable objetivo (Churn)")
ax.set_ylabel("N° de clientes")
plt.show()


NameError: name 'df' is not defined

**Lectura:** la clase está **desbalanceada** — aproximadamente **27% de los clientes
se fuga**. Esto es importante porque un modelo que simplemente prediga "No" siempre
tendría ~73% de *accuracy*, lo cual sería engañoso. Por eso no usaremos *accuracy* como
única métrica.


In [ ]:
# Churn por tipo de contrato
fig, ax = plt.subplots()
sns.barplot(data=df, x="Contract", y="Churn", ax=ax, errorbar=None,
            order=["Month-to-month", "One year", "Two year"])
ax.set_title("Tasa de churn por tipo de contrato")
ax.set_ylabel("Tasa de churn")
plt.show()


**Lectura:** los clientes con contrato **mes a mes** tienen una tasa de churn
notoriamente más alta que quienes tienen contratos de uno o dos años. Esto sugiere que
**la falta de compromiso contractual** es un fuerte predictor de fuga, y que ofrecer
incentivos para migrar a contratos anuales podría ser una palanca de retención efectiva.


In [ ]:
# Churn por metodo de pago
fig, ax = plt.subplots(figsize=(9,5))
sns.barplot(data=df, x="PaymentMethod", y="Churn", ax=ax, errorbar=None)
ax.set_title("Tasa de churn por metodo de pago")
ax.set_ylabel("Tasa de churn")
plt.xticks(rotation=20, ha="right")
plt.show()


**Lectura:** el método **electronic check** muestra una tasa de churn claramente
mayor que los métodos automáticos (tarjeta de crédito, transferencia bancaria). Los pagos
automáticos podrían estar asociados a clientes más "comprometidos" o simplemente a menor
friccion para mantenerse en el servicio.


In [ ]:
# Churn por antiguedad (tenure)
fig, ax = plt.subplots()
sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", bins=30, ax=ax)
ax.set_title("Distribucion de antiguedad (tenure) segun churn")
plt.show()


**Lectura:** el churn se concentra fuertemente en los **primeros meses** de
antigüedad. A medida que el cliente acumula más meses con la empresa, la probabilidad de
fuga decae. Esto valida la idea de reforzar la atención durante los primeros 6-12 meses
(periodo crítico de retención).


In [ ]:
# Cargos mensuales vs churn
fig, ax = plt.subplots()
sns.violinplot(data=df, x="Churn", y="MonthlyCharges", ax=ax)
ax.set_xticklabels(["No", "Yes"])
ax.set_title("Cargos mensuales segun churn")
plt.show()


**Lectura:** los clientes que se fugan tienden a pagar **cargos mensuales más
altos**. Esto, sumado a los hallazgos de contrato mes a mes, sugiere un perfil de cliente
sensible al precio que no percibe suficiente valor a largo plazo en el servicio.


In [ ]:
# Matriz de correlacion entre variables numericas y churn
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlacion entre variables numericas")
plt.show()


**Lectura:** `tenure` muestra la correlación negativa más fuerte con `Churn`
(a mayor antigüedad, menor churn), mientras que `MonthlyCharges` tiene una correlación
positiva moderada. `TotalCharges` está fuertemente correlacionado con `tenure` (es casi
una función de la antigüedad y el cargo mensual), lo que tendremos en cuenta para evitar
redundancia/multicolinealidad en el modelado.


## 3. Preprocesamiento e ingeniería de características

Construimos:
- **Nuevas variables**: número de servicios contratados, tramos de antigüedad.
- **Split train/test ANTES de ajustar cualquier transformación**, para evitar *data leakage*.
- Un **`Pipeline` de scikit-learn** con `ColumnTransformer` que aplica encoding/escalado
  ajustando únicamente con el conjunto de entrenamiento.
- Manejo del **desbalance de clases** vía `class_weight="balanced"` en los modelos.


In [ ]:
# Ingenieria de caracteristicas
service_cols = ["PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
                 "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

def count_services(row):
    count = 0
    for c in service_cols:
        val = row[c]
        if val not in ["No", "No phone service", "No internet service"]:
            count += 1
    return count

df["NumServices"] = df.apply(count_services, axis=1)

def tenure_group(t):
    if t <= 12: return "0-1 anio"
    elif t <= 24: return "1-2 anios"
    elif t <= 48: return "2-4 anios"
    else: return "4+ anios"

df["TenureGroup"] = df["tenure"].apply(tenure_group)

print(df[["NumServices", "TenureGroup"]].head())


In [ ]:
# Separamos features y target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Split estratificado train/test (80/20) -- ANTES de cualquier fit de transformaciones
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Tasa de churn train:", y_train.mean().round(3), " test:", y_test.mean().round(3))


In [ ]:
# Columnas numericas y categoricas
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "NumServices", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categorical_features),
])

print("Numericas:", numeric_features)
print("Categoricas:", categorical_features)


## 4. Selección y comparación de modelos

Comparamos **cinco modelos supervisados**: Regresión Logística, Random Forest, Gradient
Boosting, XGBoost y KNN. **Cuatro de ellos** (todos salvo KNN, que usamos como baseline
simple) incorporan **ajuste de hiperparámetros** vía `GridSearchCV`/`RandomizedSearchCV`
con **validación cruzada estratificada (5 folds)**, optimizando para **F1-score** (acorde
a la decisión de negocio de la sección 1).

Todo el preprocesamiento va dentro del `Pipeline`, por lo que el `ColumnTransformer` se
ajusta **solo con los folds de entrenamiento** en cada iteración de la validación cruzada
→ no hay *data leakage*.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = "f1"

results = {}
best_estimators = {}


In [ ]:
# 1) Regresion Logistica (con tuning)
pipe_lr = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])
param_lr = {"clf__C": [0.01, 0.1, 1, 10, 100]}
grid_lr = GridSearchCV(pipe_lr, param_lr, scoring=scoring, cv=cv, n_jobs=-1)
grid_lr.fit(X_train, y_train)
print("Mejor C:", grid_lr.best_params_, " F1 (cv):", round(grid_lr.best_score_, 3))
best_estimators["Logistic Regression"] = grid_lr.best_estimator_


In [ ]:
# 2) Random Forest (con tuning)
pipe_rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE))
])
param_rf = {
    "clf__n_estimators": [200, 400],
    "clf__max_depth": [5, 10, None],
    "clf__min_samples_leaf": [1, 3, 5],
}
grid_rf = RandomizedSearchCV(pipe_rf, param_rf, n_iter=10, scoring=scoring, cv=cv,
                              random_state=RANDOM_STATE, n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Mejores params:", grid_rf.best_params_, " F1 (cv):", round(grid_rf.best_score_, 3))
best_estimators["Random Forest"] = grid_rf.best_estimator_


In [ ]:
# 3) Gradient Boosting (con tuning)
pipe_gb = Pipeline([
    ("prep", preprocessor),
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))
])
param_gb = {
    "clf__n_estimators": [100, 200],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [2, 3, 4],
}
grid_gb = GridSearchCV(pipe_gb, param_gb, scoring=scoring, cv=cv, n_jobs=-1)
grid_gb.fit(X_train, y_train)
print("Mejores params:", grid_gb.best_params_, " F1 (cv):", round(grid_gb.best_score_, 3))
best_estimators["Gradient Boosting"] = grid_gb.best_estimator_


In [ ]:
# 4) XGBoost (con tuning)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
pipe_xgb = Pipeline([
    ("prep", preprocessor),
    ("clf", xgb.XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight,
                               random_state=RANDOM_STATE))
])
param_xgb = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [3, 4, 5],
    "clf__learning_rate": [0.05, 0.1],
}
grid_xgb = GridSearchCV(pipe_xgb, param_xgb, scoring=scoring, cv=cv, n_jobs=-1)
grid_xgb.fit(X_train, y_train)
print("Mejores params:", grid_xgb.best_params_, " F1 (cv):", round(grid_xgb.best_score_, 3))
best_estimators["XGBoost"] = grid_xgb.best_estimator_


In [ ]:
# 5) KNN (baseline, sin tuning extenso)
pipe_knn = Pipeline([
    ("prep", preprocessor),
    ("clf", KNeighborsClassifier(n_neighbors=15))
])
pipe_knn.fit(X_train, y_train)
best_estimators["KNN (baseline)"] = pipe_knn
print("KNN entrenado (baseline, sin tuning de hiperparametros)")


In [ ]:
# Tabla comparativa de modelos en TEST set
rows = []
for name, model in best_estimators.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

comparison_df = pd.DataFrame(rows).set_index("Modelo").round(3).sort_values("F1", ascending=False)
comparison_df


**Lectura de la tabla comparativa:** se muestra el desempeño de los 5 modelos en el
conjunto de prueba (no visto durante el tuning). El modelo con mejor combinación de F1 y
ROC-AUC se selecciona como **modelo final** para las secciones de evaluación e
interpretación siguientes.


In [ ]:
best_model_name = comparison_df["F1"].idxmax()
best_model = best_estimators[best_model_name]
print("Modelo final seleccionado:", best_model_name)


## 5. Evaluación del modelo final

Analizamos en detalle el **modelo final** seleccionado: matriz de confusión, curva ROC y
el efecto de mover el **umbral de decisión** sobre el equilibrio precision/recall.


In [ ]:
y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_best, target_names=["No Churn", "Churn"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusion
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=["No Churn", "Churn"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Matriz de confusion - {best_model_name}")

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba_best)
auc = roc_auc_score(y_test, y_proba_best)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
axes[1].plot([0,1],[0,1], linestyle="--", color="gray")
axes[1].set_xlabel("Tasa de falsos positivos")
axes[1].set_ylabel("Tasa de verdaderos positivos")
axes[1].set_title("Curva ROC")
axes[1].legend()

plt.tight_layout()
plt.show()


**Lectura:** la matriz de confusión muestra cuántos clientes que realmente se
fugaron fueron correctamente identificados (verdaderos positivos) frente a los que el
modelo no detectó (falsos negativos, el error más costoso según la sección 1). La curva
ROC con AUC alto indica que el modelo distingue bien entre ambas clases en distintos
umbrales de decisión.


In [ ]:
# Efecto del umbral de decision sobre precision/recall
precisions, recalls, thr = precision_recall_curve(y_test, y_proba_best)

fig, ax = plt.subplots()
ax.plot(thr, precisions[:-1], label="Precision")
ax.plot(thr, recalls[:-1], label="Recall")
ax.axvline(0.5, color="gray", linestyle="--", label="Umbral por defecto (0.5)")
ax.set_xlabel("Umbral de decision")
ax.set_ylabel("Score")
ax.set_title("Precision y Recall en funcion del umbral")
ax.legend()
plt.show()


**Lectura y decisión de negocio:** bajar el umbral por debajo de 0.5 aumenta el
**Recall** (detectamos a más clientes que se van) a costa de la **Precision** (más falsos
positivos, es decir, más campañas de retención "desperdiciadas"). Dado que en la sección 1
concluimos que el falso negativo es más costoso, **recomendamos operar con un umbral algo
más bajo que 0.5** (por ejemplo, ~0.35-0.40) para priorizar la detección de clientes en
riesgo, siempre que el costo de la campaña de retención sea razonable frente al valor de
vida del cliente.


## 6. Interpretación del modelo

Abrimos la "caja negra" del modelo final usando **importancia de variables** (basada en
permutación, válida para cualquier modelo) y **SHAP**, que además nos dice la dirección del
efecto de cada variable.


In [ ]:
# Importancia por permutacion (model-agnostic)
perm = permutation_importance(best_model, X_test, y_test, scoring="f1",
                               n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)

importances = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8,6))
importances.head(12).plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title(f"Importancia de variables (permutacion) - {best_model_name}")
ax.set_xlabel("Caida promedio en F1 al permutar la variable")
plt.show()


In [ ]:
# SHAP - usamos el modelo XGBoost (entrenado de forma independiente sobre datos
# ya transformados) para obtener un resumen interpretable, ya que SHAP es mas directo
# de aplicar sobre modelos basados en arboles
prep_fitted = best_estimators["XGBoost"].named_steps["prep"]
clf_fitted = best_estimators["XGBoost"].named_steps["clf"]

X_test_transformed = prep_fitted.transform(X_test)
feature_names = prep_fitted.get_feature_names_out()
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

explainer = shap.TreeExplainer(clf_fitted)
shap_values = explainer.shap_values(X_test_df)

shap.summary_plot(shap_values, X_test_df, show=False, max_display=12)
plt.tight_layout()
plt.show()


**Lectura:** las variables más influyentes suelen ser la **antigüedad (tenure)**, el
**tipo de contrato** (mes a mes vs anual) y los **cargos mensuales**. Valores bajos de
`tenure` y contratos mes a mes empujan la predicción hacia "churn", confirmando los
patrones detectados en el EDA, ahora con respaldo desde la perspectiva del modelo.


## 7. Análisis complementario: segmentación de clientes (no supervisado)

Como análisis adicional, aplicamos **clustering (K-Means)** sobre variables numéricas
estandarizadas, reducidas a 2 componentes con **PCA** para visualización, con el fin de
identificar **perfiles de clientes** más allá de la etiqueta de churn.


In [ ]:
cluster_features = ["tenure", "MonthlyCharges", "TotalCharges", "NumServices"]
X_cluster = StandardScaler().fit_transform(df[cluster_features])

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(X_cluster)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster)

fig, ax = plt.subplots()
scatter = ax.scatter(X_pca[:,0], X_pca[:,1], c=clusters, cmap="tab10", alpha=0.6, s=15)
ax.set_title("Segmentacion de clientes (K-Means + PCA)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.colorbar(scatter, label="Cluster")
plt.show()

df_clusters = df.copy()
df_clusters["Cluster"] = clusters
df_clusters.groupby("Cluster")[["tenure","MonthlyCharges","TotalCharges","NumServices","Churn"]].mean().round(2)


**Lectura:** los clústeres revelan perfiles distintos de clientes (por ejemplo,
clientes nuevos con cargos altos y pocos servicios vs. clientes antiguos con muchos
servicios contratados). El clúster con **mayor tasa de churn promedio** coincide con el
perfil de bajo *tenure* y alto cargo mensual identificado en el EDA y en SHAP, reforzando
la consistencia de los hallazgos.


## 8. Conclusiones y recomendaciones de negocio

**Principales hallazgos:**

1. La tasa de churn global es de ~27%. El riesgo se concentra fuertemente en los
   **primeros 12 meses** de antigüedad.
2. El **tipo de contrato** es el predictor más fuerte: contratos mes a mes elevan
   sustancialmente el riesgo de fuga frente a contratos de uno o dos años.
3. El **método de pago electronic check** y los **cargos mensuales altos** están
   asociados a mayor churn.
4. El modelo final seleccionado logra un buen balance entre Recall y Precision en el
   conjunto de prueba (ver tabla comparativa), siendo apto para apoyar decisiones de
   retención.

**Recomendaciones accionables:**

- **Incentivar la migración** de clientes mes a mes hacia contratos anuales (ej. descuento
  por compromiso de 12 meses), especialmente durante los primeros meses de vida del
  cliente, que es la ventana de mayor riesgo.
- **Promover medios de pago automáticos** (tarjeta/transferencia) sobre electronic check,
  reduciendo friccion y posiblemente la tasa de abandono asociada.
- **Focalizar campañas de retención** usando las probabilidades del modelo: priorizar a
  clientes con score de churn alto y alto valor (`MonthlyCharges`/`TotalCharges`), donde el
  retorno de una campaña de retención es mayor.
- Usar el **umbral de decisión ajustado** (sección 5) para balancear el costo de
  campañas de retención frente al costo de perder clientes.

**Limitaciones y mejoras futuras:**

- El dataset es una **foto estática** (no longitudinal); no captura cambios de
  comportamiento en el tiempo.
- Podría explorarse el uso de **redes neuronales** o *stacking* de modelos para mejorar
  el desempeño marginal.
- Sería valioso incorporar **datos de interacción con soporte/quejas** si estuvieran
  disponibles, ya que probablemente sean predictores adicionales relevantes de churn.
